# Layer 2 — NWS Forecast Calibration

**Goal:** quantify how often, by how much, and in which direction NWS forecasts of the LAX daily high diverge from the actual high. Use that distribution to convert any new NWS point forecast into a calibrated probability distribution over integer °F strikes.

**Data:** ~12 months of historical PFM (Point Forecast Matrix) issuances from the Los Angeles/Oxnard WFO, pulled from the Iowa State NWS text product archive. Each issuance includes the forecast high for the issuance day plus the next 6 days, so one calendar day produces ~28 (forecast, target) pairs at varying lead times.

**Method:** empirical (not Normal) residual distribution per lead-time bucket. We shift the residual cloud by the new forecast value to construct a calibrated distribution.

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lax_forecast.data import load_lax_history
from lax_forecast.climatology import build_climatology_from_loaded
from lax_forecast.calibration import (
    load_pfm_archive, build_residuals_table, ForecastCalibrator,
    build_default_calibrator,
)
from lax_forecast.nws import get_daily_high

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

## 1. Load forecasts and actuals

PFM cache is built by `scripts/backfill_pfm.py`. If it doesn't exist or is small, run:

```bash
python scripts/backfill_pfm.py --days 365 --quiet
```

In [ ]:
actuals = load_lax_history().df['tmax_f']
forecasts = load_pfm_archive()

print(f'Actuals: {len(actuals):,} days ({actuals.index.min().date()} → {actuals.index.max().date()})')
print(f'Forecasts: {len(forecasts):,} rows ({forecasts["target_date"].min()} → {forecasts["target_date"].max()})')
print(f'  unique issuances: {forecasts["product_id"].nunique():,}')
print(f'  unique target dates: {forecasts["target_date"].nunique():,}')
forecasts.head()

## 2. Compute residuals

Residual = forecast − actual. Positive means NWS over-forecast (the high came in lower than predicted).

In [ ]:
residuals = build_residuals_table(forecasts, actuals)
print(f'Joined rows: {len(residuals):,}')
print(f'\nOverall residual stats (forecast − actual):')
residuals['residual'].describe().to_frame().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(residuals['residual'], bins=np.arange(-15, 16) - 0.5, edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='k', lw=1)
axes[0].axvline(residuals['residual'].mean(), color='red', lw=1.5, label=f"mean = {residuals['residual'].mean():+.2f}°F")
axes[0].set_xlabel('Residual (forecast − actual), °F')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of all forecast residuals')
axes[0].legend()

by_lead = residuals.groupby('lead_bucket', observed=True)['residual'].agg(['count','mean','std'])
axes[1].errorbar(range(len(by_lead)), by_lead['mean'], yerr=by_lead['std'], fmt='o-', capsize=4)
axes[1].axhline(0, color='k', lw=0.6)
axes[1].set_xticks(range(len(by_lead)))
axes[1].set_xticklabels(by_lead.index, rotation=45, ha='right')
axes[1].set_xlabel('Lead time')
axes[1].set_ylabel('Mean residual ± 1 SD (°F)')
axes[1].set_title('Forecast bias & spread by lead time')
plt.tight_layout()

In [ ]:
by_lead

**Read the table carefully.**

- `mean` is the bias: if positive, NWS systematically over-forecasts at this lead time. Subtract it from new forecasts for a quick bias correction.
- `std` is the spread: standard deviation of residuals at this lead time. This goes into the variance of our calibrated distribution.
- For Kalshi: same-day trading happens with lead times of 0–24h. That's the most important bucket; the others matter for multi-day strikes.

## 3. Seasonal residual structure

LAX bias varies by month — marine-layer months (May–July) tend to have a positive bias because NWS underweights stratus persistence.

In [ ]:
near_term = residuals[residuals['lead_hours'].between(0, 36)]
by_month = near_term.groupby('month')['residual'].agg(['count','mean','std','median'])
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_month.index, by_month['mean'], yerr=by_month['std'], capsize=4, alpha=0.85)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Mean residual ± 1 SD (°F)')
ax.set_title('Day-ahead (≤36h) forecast bias by month')
plt.tight_layout()
by_month

## 4. Fit the calibrator

In [ ]:
cal = ForecastCalibrator(residuals)
cal.summary()

## 5. Climatology vs Calibrated NWS vs Raw NWS — today

Pull today's NWS gridpoint forecast, calibrate it, and compare to the climatology prior from notebook #01.

In [ ]:
nws = get_daily_high()
today = pd.Timestamp.today().normalize()

# The default lead for the daily forecast is ~6-12h (issued morning, max ~2pm).
lead_hours = 6
cal_dist = cal.calibrate(nws.high_f, lead_hours=lead_hours)

actuals_full = load_lax_history().df
clim = build_climatology_from_loaded(actuals_full, window_days=15, recency_halflife_years=10)
clim_dist = clim.distribution(today)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(clim_dist.temps_f - 0.2, clim_dist.probs, width=0.4, alpha=0.55, label=f'Climatology (mean {clim_dist.mean:.1f}°F)')
ax.bar(cal_dist.temps_f + 0.2, cal_dist.probs, width=0.4, alpha=0.75, label=f'Calibrated NWS @ {lead_hours}h (mean {cal_dist.mean:.1f}°F)')
ax.axvline(nws.high_f, color='red', lw=2.5, label=f'Raw NWS forecast ({nws.high_f}°F)')
ax.set_xlabel(f'TMAX (°F) on {today.date()}')
ax.set_ylabel('Probability')
ax.set_title(f'Layer 1 (climatology) vs Layer 2 (calibrated NWS) — {today.date()}')
ax.legend()
plt.tight_layout()

print(f'Raw NWS forecast:        {nws.high_f}°F  ({nws.short_forecast})')
print(f'Calibrated mean:         {cal_dist.mean:.2f}°F  (bias-correction = {nws.high_f - cal_dist.mean:+.2f}°F)')
print(f'Calibrated 90% CI:       [{cal_dist.quantile(0.05):.0f}, {cal_dist.quantile(0.95):.0f}]°F')
print(f'Climatology mean:        {clim_dist.mean:.2f}°F  (90% CI [{clim_dist.quantile(0.05):.0f}, {clim_dist.quantile(0.95):.0f}])')

## 6. Strike probabilities — climatology vs calibrated NWS

In [ ]:
strikes = np.arange(int(cal_dist.mean) - 8, int(cal_dist.mean) + 9)
table = pd.DataFrame({
    'strike_°F': strikes,
    'P(>strike) | clim': [round(clim_dist.p_greater_than(s), 3) for s in strikes],
    'P(>strike) | calibrated': [round(cal_dist.p_greater_than(s), 3) for s in strikes],
    'edge': [round(cal_dist.p_greater_than(s) - clim_dist.p_greater_than(s), 3) for s in strikes],
}).set_index('strike_°F')
table

The `edge` column is what matters for trading: if the market is pricing strikes off climatology (the lazy prior) but our calibrated distribution differs, that gap is a mispricing.

## What's next

1. **Lead-time-aware backtesting** — replay the last 12 months pretending we only knew the NWS forecast at each historical time, score against actuals.
2. **Probability calibration validation** — reliability diagram: when we say P=0.30, does it actually happen 30% of the time?
3. **Layer 3 — HRRR + marine-layer regime** — split the residual distribution conditional on stratus indicators.
4. **Strike-by-strike comparison vs live Kalshi orderbook** — Layer 5.